# Baseline Model training

Baseline classifier that labels 2024 trademarks as **green**, **mixed** or **non-green**, and ranks the unlabelled ones by how likely they are to be green.

| Input | Encoding |
|---|---|
| Mark name + goods/services text | TF-IDF (words and word pairs) |
| Nice codes (1-45) | Multi-hot: one 0/1 column per class |
| Vienna codes, main category (1-29) | Multi-hot: one 0/1 column per category |
| All three combined | Logistic regression |

**Labels are not available yet.** The notebook expects them in `../../labels/labels.csv` (outside the repo), with two columns:

```
Application Number,label
1505797,non-green
1990531,mixed
```

`label` must be one of `green`, `mixed`, `non-green`. Until that file exists, set `USE_DUMMY_LABELS = True` to run the whole notebook on **random** labels. That only checks the code runs: the scores will be at chance level and mean nothing.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, average_precision_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline

ROOT = Path.cwd().parents[1]                 # green_innovation_screening_all/
POOL_FILE = ROOT / "pool" / "TM_registered_2024.csv"
LABELS_FILE = ROOT / "labels" / "labels.csv"
ARTIFACTS = ROOT / "artifacts"               # trained models and rankings, kept off GitHub

CLASSES = ["green", "mixed", "non-green"]
USE_DUMMY_LABELS = False                     # True = random labels, only to test the code runs
RANDOM_STATE = 42

## Step 1 - Load the trademarks and build the three inputs

- Keep ordinary trademarks only (`Trademark Class Code == "1"`), as in the cleaning notebook.
- **`text`**: the mark name plus all its goods/services statements. The ` || ` statement separator becomes a full stop so it doesn't turn into a token.
- **`nice`**: the Nice codes as they are, e.g. `03;05;41`.
- **`vienna`**: only the main category of each Vienna code (`26.4.2` becomes `26`), with repeats removed. Most marks have no Vienna code, so this is often empty. Empty means "not coded", not "no logo".

In [ ]:
pool = pd.read_csv(POOL_FILE, sep="|", encoding="utf-8", dtype=str, index_col="Application Number")
pool = pool.loc[pool["Trademark Class Code"] == "1"]


def vienna_categories(codes):
    """'26.4.2;26.4.5;29.1.1' -> '26;29' (main categories, sorted, no repeats)."""
    if pd.isna(codes):
        return ""
    return ";".join(sorted({c.split(".")[0] for c in codes.split(";") if c}, key=int))


X_all = pd.DataFrame(
    {
        "text": pool["Mark Significant Description"].fillna("") + " . "
                + pool["Goods Services Description"].fillna("").str.replace(" || ", " . ", regex=False),
        "nice": pool["Nice Code"].fillna(""),
        "vienna": pool["Vienna Code"].map(vienna_categories),
    },
    index=pool.index,
)
print(f"{len(X_all):,} trademarks")
X_all.head()

## Step 2 - Load the labels

Labels are matched to trademarks by `Application Number`. The cell stops with an error if a label is misspelled or the file is missing, unless `USE_DUMMY_LABELS = True`.

In [ ]:
if LABELS_FILE.exists():
    labels = pd.read_csv(LABELS_FILE, dtype=str).set_index("Application Number")["label"]
    labels = labels.str.strip().str.lower()
    unknown = set(labels.unique()) - set(CLASSES)
    if unknown:
        raise ValueError(f"Unknown labels {unknown}; allowed: {CLASSES}")
    missing = ~labels.index.isin(X_all.index)
    if missing.any():
        print(f"Warning: {missing.sum()} labelled application numbers are not in the pool and are ignored")
    labels = labels[~missing]
elif USE_DUMMY_LABELS:
    # Random labels, only to check the notebook runs end to end. Scores will be meaningless.
    rng = np.random.default_rng(RANDOM_STATE)
    sample = X_all.sample(3000, random_state=RANDOM_STATE).index
    labels = pd.Series(rng.choice(CLASSES, size=len(sample), p=[0.1, 0.2, 0.7]), index=sample)
    print("USING RANDOM DUMMY LABELS - results below are not real")
else:
    raise FileNotFoundError(f"No labels at {LABELS_FILE}. Add them, or set USE_DUMMY_LABELS = True.")

X, y = X_all.loc[labels.index], labels
y.value_counts()

## Step 3 - Turn the inputs into numbers

`ColumnTransformer` applies one encoder per input and puts the results side by side in one large, mostly-zero (sparse) matrix:

- **TF-IDF on `text`** gives each word or word pair a weight: higher when it is frequent in this mark's text but rare across marks. Settings:
  - `ngram_range=(1, 2)`: single words and word pairs, so "solar panels" is a feature as well as "solar".
  - `min_df=2`: ignore terms that appear in only one mark (typos, one-off words).
  - `max_df=0.9`: ignore terms that appear in over 90% of marks (they say nothing).
  - `sublinear_tf=True`: a word repeated 1,000 times (a catalogue like ENERJO's) counts only a little more than one repeated 10 times.
  - `strip_accents="unicode"`: makes French and English spellings line up (é becomes e).
- **Multi-hot on `nice` and `vienna`**: `CountVectorizer` splits on `;` and gives one 0/1 column per code. A fixed vocabulary (all 45 Nice classes, all 29 Vienna categories) keeps the columns the same even if a code never appears in the training labels.

The encoders are inside the pipeline, so they are fitted on training data only. Nothing from the test set leaks into the vocabulary.

In [ ]:
NICE_CODES = [f"{i:02d}" for i in range(1, 46)]
VIENNA_CATEGORIES = [str(i) for i in range(1, 30)]


def multi_hot(vocabulary):
    """One 0/1 column per code in a ';'-separated string."""
    return CountVectorizer(token_pattern=r"[^;]+", lowercase=False, binary=True, vocabulary=vocabulary)


features = ColumnTransformer([
    ("text", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9, sublinear_tf=True,
                             strip_accents="unicode", max_features=100_000), "text"),
    ("nice", multi_hot(NICE_CODES), "nice"),
    ("vienna", multi_hot(VIENNA_CATEGORIES), "vienna"),
])

## Step 4 - Choose the model

**Logistic regression** is the standard baseline for TF-IDF. It is fast on large sparse matrices, and each feature gets one readable weight per class (step 7). `class_weight="balanced"` counts mistakes on rare classes more, because green marks will probably be a small minority. Without it, the model could ignore green and still look accurate.

A **dummy model** that always predicts the most common class is trained next to it. The real model has to beat it clearly to be worth anything.

In [ ]:
model = Pipeline([
    ("features", features),
    ("classifier", LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000)),
])
dummy = DummyClassifier(strategy="most_frequent")

## Step 5 - Train and evaluate on a held-out test set

- **Split:** 80% train / 20% test, stratified so each split has the same share of green / mixed / non-green.
- **Macro F1** averages the F1 score of the three classes equally, so the rare green class counts as much as the common one. Use it as the headline metric. Plain accuracy is misleading when classes are unbalanced.
- **Average precision for green** measures ranking quality: how high the truly green marks come when all test marks are sorted by predicted probability of green. It is the closest metric to the project goal of ranking candidates. A random ranking scores about the share of green marks.
- **5-fold cross-validation** repeats the training on five different splits. The spread shows how stable the score is, which matters when there are few labels.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
model.fit(X_train, y_train)
dummy.fit(X_train, y_train)

pred = model.predict(X_test)
print("Baseline (logistic regression)\n", classification_report(y_test, pred, labels=CLASSES, zero_division=0))
print("Dummy (always most common class)\n",
      classification_report(y_test, dummy.predict(X_test), labels=CLASSES, zero_division=0))

green_col = list(model.classes_).index("green")
p_green = model.predict_proba(X_test)[:, green_col]
print(f"Average precision for green: {average_precision_score(y_test == 'green', p_green):.3f} "
      f"(random ranking: {(y_test == 'green').mean():.3f})")

ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=CLASSES, cmap="Blues", colorbar=False)
plt.title("Confusion matrix (test set)")
plt.show()

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(model, X, y, cv=cv, scoring="f1_macro")
print(f"5-fold macro F1: {scores.mean():.3f} +/- {scores.std():.3f}  (folds: {np.round(scores, 3)})")

## Step 6 - Look at what the model learned

The words and codes with the largest weight for each class. With real labels, the **green** list should show terms like *solar*, *recycling* or *biodegradable*. If it shows unrelated words, the labels or the features need checking. With dummy labels the lists are noise.

In [ ]:
names = model.named_steps["features"].get_feature_names_out()
coefs = model.named_steps["classifier"].coef_
top = {
    cls: [names[i].split("__", 1)[1] + ("" if names[i].startswith("text__") else f" ({names[i].split('__')[0]})")
          for i in np.argsort(coefs[k])[::-1][:20]]
    for k, cls in enumerate(model.classes_)
}
pd.DataFrame(top)

## Step 7 - Retrain on all labels and rank every trademark

The final model is trained on all labelled marks and scores all 61,631 trademarks. Marks are ranked by `green_score = P(green) + 0.5 * P(mixed)`, so marks that are partly green come after clearly green ones but before non-green ones. The ranking and the model are saved to `../../artifacts/`, outside the repo.

In [ ]:
model.fit(X, y)

proba = pd.DataFrame(model.predict_proba(X_all), index=X_all.index,
                     columns=[f"P({c})" for c in model.classes_])
ranking = pool[["Mark Significant Description", "Nice Code"]].join(proba)
ranking["green_score"] = ranking["P(green)"] + 0.5 * ranking["P(mixed)"]
ranking["labelled"] = ranking.index.isin(y.index)
ranking = ranking.sort_values("green_score", ascending=False)

ARTIFACTS.mkdir(exist_ok=True)
ranking.to_csv(ARTIFACTS / "baseline_ranking_2024.csv", sep="|", encoding="utf-8")
joblib.dump(model, ARTIFACTS / "baseline_model.joblib")
print(f"Saved ranking and model to {ARTIFACTS}")
ranking.head(20)